# GenRec-Food on Kaggle (Yelp + Qwen2.5-0.5B)

Faithful small-scale reproduction of Netflix's GenRec: LLM backbone + catalog-aware
ranking head, two-phase training, three losses, prefill-only scoring.

**Setup:** add the *Yelp Dataset* as input data, enable GPU (T4/P100), then run top to bottom.

### 1. Get the source
Clone/copy the `src/` package. If you uploaded this repo as a Kaggle dataset, adjust the path.

In [ ]:
import sys, os
# Option A: repo uploaded as a Kaggle dataset input
SRC = "/kaggle/input/genrec-food/src"
# Option B: pip/git clone your repo, or paste src/*.py into /kaggle/working/src
if not os.path.exists(SRC):
    SRC = "/kaggle/working/src"
sys.path.insert(0, SRC)
print("src:", SRC, "exists:", os.path.exists(SRC))

In [ ]:
!pip -q install "transformers>=4.44" "torch>=2.2" tqdm

### 2. Load Yelp restaurants (one metro) → catalog + sequences

In [ ]:
from data import load_yelp
YELP_DIR = "/kaggle/input/yelp-dataset"   # dir with yelp_academic_dataset_business.json + review.json
ds = load_yelp(YELP_DIR, city=None, min_user_interactions=5, max_users=20000)
print("items:", ds.num_items, "users:", len(ds.sequences), "eval users:", len(ds.test))

### 3. Baselines (the bar to beat)

In [ ]:
from baselines import PopularityBaseline, ItemKNNBaseline
from eval import evaluate, pretty
exclude = {u: {it.item_id for it in ds.train_hist[u]} for u in ds.test}
for name, m in [("Popularity", PopularityBaseline(ds, ds.train_hist)),
                ("ItemKNN", ItemKNNBaseline(ds, ds.train_hist))]:
    print(pretty(name, evaluate(m.scorer, ds.test, ds.num_items, ks=(10,), exclude=exclude)))

### 4. Build GenRec on a real backbone

In [ ]:
import torch
from model import GenRec, build_backbone
device = "cuda" if torch.cuda.is_available() else "cpu"
lm, tok, hid = build_backbone("Qwen/Qwen2.5-0.5B", tiny=False)
model = GenRec(lm, hid, num_items=ds.num_items, scorer="dot").to(device)
print("device:", device, "hidden:", hid)

### 5. Phase 1 — domain adaptation (causal LM)

In [ ]:
from train import phase1_adapt
phase1_adapt(model, tok, ds, device, epochs=1, lr=5e-5, budget=10, batch_size=16)

### 6. Phase 2 — ranking post-training (three losses)

In [ ]:
from train import phase2_rank
phase2_rank(model, tok, ds, device, epochs=2, lr=1e-4, budget=10,
            batch_size=16, n_neg=8, weights=(1.0, 0.1, 0.5))

### 7. Evaluate GenRec (prefill-only scoring path)

In [ ]:
from train import GenRecScorer
gr = GenRecScorer(model, tok, ds, device, budget=10)
print(pretty("GenRec", evaluate(gr.scorer, ds.test, ds.num_items, ks=(10,), exclude=exclude)))

### 8. Ablations (mirroring the blog)
Phase-1 on/off, each loss on/off, context ⅓ — each is a fresh short run.
For speed on Kaggle, reduce `max_users` above or `p2-epochs` here.

In [ ]:
from ablations import run
common = dict(ds=ds, device=device, model_name="Qwen/Qwen2.5-0.5B", tiny=False,
              budget=10, p1_epochs=1, p2_epochs=2)
print("no Phase-1 :", run("no_p1", do_phase1=False, **common))
print("no LM loss :", run("no_lm", use_lm=False, **common))
print("no reward  :", run("no_rew", use_reward=False, **common))
print("context 1/3:", run("ctx13", **{**common, "budget": 3}))